# Пара 4. Zipf и частоты на **своём** тексте

Перенос процедуры пары 1 на материал диссертации (или Чехова, если файлов ещё нет).


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


### Свой текст диссертации
В Colab: значок папки слева → загрузите `.txt` (UTF-8) фрагментов/глав, которые идут в магистерскую.
Дальше в коде поставьте `MODE = "thesis"` и перечислите имена файлов.


In [ ]:
import re, requests
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

MODE = "demo"  # или "thesis"
THESIS_FILES = ["glava1.txt"]
CHEKHOV = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/chekhov-rasskazy/txt/"

def load_texts():
    out = {}
    if MODE == "thesis":
        for f in THESIS_FILES:
            out[Path(f).stem] = Path(f).read_text(encoding="utf-8")
    else:
        for title, fn in {"Дама с собачкой": "dama-s-sobachkoj.txt", "Крыжовник": "kryzhovnik.txt"}.items():
            r = requests.get(CHEKHOV + fn, timeout=60); r.encoding = "utf-8"
            out[title] = r.text
    return out

def tokenize(text):
    return re.findall(r"[А-Яа-яЁёA-Za-z]+", text.lower())

texts = load_texts()
# СВОИ поля под гипотезу диссертации:
FIELDS = {
    "поле_A": ["укажите", "стем1"],
    "поле_B": ["укажите", "стем2"],
}
# демо-подсказка для Чехова, если не задали свои:
if MODE == "demo":
    FIELDS = {
        "любовь": ["люб", "страсть", "сердц"],
        "скука_быт": ["скуч", "обыден", "футляр", "крыжов"],
        "смерть_болезнь": ["смерть", "болезн", "похорон"],
    }

rows = []
for title, text in texts.items():
    toks = tokenize(text)
    n = max(len(toks), 1)
    row = {"текст": title, "слов": n}
    for name, stems in FIELDS.items():
        c = sum(1 for t in toks if any(t.startswith(s) for s in stems))
        row[f"{name}_на_1000"] = round(1000 * c / n, 2)
    rows.append(row)
df = pd.DataFrame(rows)
df


In [ ]:
cols = [c for c in df.columns if c.endswith("_на_1000")]
df.set_index("текст")[cols].plot(kind="bar", figsize=(9, 4), rot=20)
plt.ylabel("на 1000"); plt.title("Поля под ваш вопрос"); plt.tight_layout(); plt.show()

title0 = next(iter(texts))
ranks = sorted(Counter(tokenize(texts[title0])).values(), reverse=True)[:100]
plt.figure(figsize=(7, 4))
plt.loglog(range(1, len(ranks)+1), ranks, "o", markersize=3)
plt.title(f"Zipf · {title0}"); plt.xlabel("ранг"); plt.ylabel("частота")
plt.tight_layout(); plt.show()


## Сдать: список стеммов (почему такие) + график + цитата-проверка.
